In [1]:
# Setup
from pyspark.sql import SparkSession
from pyspark.sql.types import (
StructType, StructField, StringType, IntegerType,
DoubleType, LongType, DateType, TimestampType
)
#master node IP 192.168.2.47 is used everywhere
#Change cores (set to 1 for test)
spark = SparkSession.builder \
.appName("Test") \
.master("spark://192.168.2.47:7077") \
.config("spark.dynamicAllocation.enabled", "true") \
.config("spark.shuffle.service.enabled", "false") \
.config("spark.dynamicAllocation.shuffleTracking.enabled", "true") \
.config("spark.cores.max", 2) \ #where core is changed
.getOrCreate()

# Defining an explicit schema
reddit_schema = StructType([
StructField("author", StringType(), True),
StructField("body", StringType(), True),
StructField("normalizedBody", StringType(), True),
StructField("content", StringType(), True),
StructField("content_len", LongType(), True),               # LongType is a 64-bit signed integer (IntegerType is 32-bit)
StructField("summary", StringType(), True),
StructField("summary_len", LongType(), True),
StructField("id", StringType(), True),
StructField("subreddit", StringType(), True),
StructField("subreddit_id", StringType(), True),
StructField("title", StringType(), True),
])

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/10 10:33:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/10 10:33:43 WARN ResourceProfileManager: Neither executor cores is set for resource profile, nor spark.executor.cores is explicitly set, you may get more executors allocated than expected. It's recommended to set executor cores explicitly. Please check SPARK-30299 for more details.
26/03/10 10:33:43 WARN StandaloneSchedulerBackend: Dynamic allocation enabled without spark.executor.cores explicitly set, you may get more executors allocated than expected. It's recommended to set spark.executor.cores explicitly. Please check SPARK-30299 for more details.


In [2]:
# Extracting relevant data
# Starting with extracting top 10 most common profanity-words
from pyspark.sql.functions import (
col, to_timestamp, when, lit, array,
explode, window, count, desc
)

import time

swear_words = ["the", "fuck", "shit", "asshole", "cunt", "cock", "retard"]       # Examples of swear-words, the testlist

# Reading data
hdfs = "hdfs://192.168.2.47:9000"

#df_raw = spark.read.json(f"{hdfs}/dataset/corpus-webis-tldr-17.json")
#df_raw.show(1, truncate=False)
#df_raw.printSchema()

# Change correct file location
df = spark.read \
.format("json") \
.schema(reddit_schema) \
.load(f"{hdfs}/dataset/corpus-webis-tldr-17.json")   # ADD CORRECT PATH

df.show(1, truncate=False)

# C.2 -- Apply the identical pipeline from Part B (no withWatermark)
#batch_filtered = batch_df.filter(col("retweeted_status").isNull())     # FRÅGA: Vill vi filtrera bort något?

# No batching needed (?)
#batch_parsed = batch_filtered.withColumn(
#"ts", to_timestamp(col("created_at"), "EEE MMM dd HH:mm:ss Z yyyy")
#)

swear_word_array = array(*[
when(col("Content").rlike(rf"(?i)\b{s}\b"), lit(s)) for s in swear_words        # Only finds isolated swear-words, e.g. not "momfuck"
])

# Check swear_word (unbatched)
swear_word_df = (df
.withColumn("Swear_word", explode(swear_word_array))
.filter(col("Swear_word").isNotNull())
)

# Counts number of swear words for entire dataseet
swear_words_results = (swear_word_df
.groupBy("Swear_word")
.agg(count("*").alias("Count"))
)

# C.4 -- Time the execution
start = time.time()
swear_words_results.show(50, truncate=False) #Put truncate to true, switch to false for entire printout
print(f"Time to run job (with how many workers?): {time.time() - start:.1f}s")


# Just some print options
#df.printSchema()
#df.show(5)   # Show first 5 rows
#print(f"Number of partitions: {df.rdd.getNumPartitions()}")
#print(f"Number of rows: {df.count()}")

+----------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

[Stage 1:======================================================>(117 + 1) / 118]

+----------+-------+
|Swear_word|Count  |
+----------+-------+
|retard    |4115   |
|shit      |316362 |
|cock      |8221   |
|the       |3132934|
|asshole   |43678  |
|cunt      |7149   |
|fuck      |209487 |
+----------+-------+

Time to run job (with how many workers?): 592.8s


In [3]:
spark.stop()